# 01 – Exploratory Data Analysis (EDA)

In this notebook we explore the IEEE-CIS Fraud Detection dataset.

Goals:

- Understand the size and shape of the data.
- Inspect basic distributions of key features.
- Look at the target column `isFraud` and its imbalance.
- Look for simple patterns that may indicate fraud.

Exploratory analysis helps us build intuition and guides feature engineering.


In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
sns.set(style="whitegrid")

In [ ]:
data_dir = "../data/raw"

train_transaction = pd.read_csv(os.path.join(data_dir, "train_transaction.csv"))
train_identity = pd.read_csv(os.path.join(data_dir, "train_identity.csv"))

train_transaction.shape, train_identity.shape

In [ ]:
# We merge on TransactionID just like in src/preprocess.py

train_merged = train_transaction.merge(
    train_identity, how="left", on="TransactionID"
)

train_merged.shape

In [ ]:
# Quick look at columns and data types
train_merged.info()

In [ ]:
# Plot distribution of target (isFraud)
fraud_counts = train_merged["isFraud"].value_counts(normalize=True)

print("Fraction of fraudulent transactions:", fraud_counts.get(1, 0))
print("Fraction of non-fraud transactions:", fraud_counts.get(0, 0))

sns.countplot(x="isFraud", data=train_merged)
plt.title("Target Distribution – isFraud")
plt.show()

In [ ]:
# Visualise transaction amount for fraud and non-fraud
plt.figure()
sns.histplot(
    data=train_merged,
    x="TransactionAmt",
    hue="isFraud",
    bins=100,
    element="step",
    stat="density",
    common_norm=False,
)
plt.xlim(0, 2000)
plt.title("Transaction Amount distribution by Fraud label (capped at 2000)")
plt.show()

In [ ]:
# Select a small subset of numeric columns for correlation
numeric_cols = ["TransactionAmt", "TransactionDT", "card1", "card2", "isFraud"]
subset = train_merged[numeric_cols].dropna()

corr = subset.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix (small subset)")
plt.show()

## Summary

From this quick EDA we learned:

- The dataset is heavily imbalanced, with only a small fraction of transactions labeled as fraud.
- Transaction amounts differ between fraudulent and non-fraudulent transactions, although there is overlap.
- Some card features and transaction time may have useful relationships with fraud.

Next steps:

- Design feature engineering based on our understanding.
- Handle high-cardinality categorical features.
- Prepare the data pipeline for model training in the next notebook.
